# Analisis Comparativo de Modelos SLM

Este notebook visualiza y compara los resultados del benchmark de diferentes modelos SLM
para el asistente de normatividad de Uninorte.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Cargar resultados

In [ ]:
# Buscar el archivo de resultados mas reciente
results_dir = Path('results')
raw_files = sorted(results_dir.glob('*_raw_results.json'))
summary_files = sorted(results_dir.glob('*_summary.json'))

if not raw_files:
    print('No se encontraron resultados. Ejecuta primero:')
    print('  python -m benchmark.run_benchmark')
else:
    print(f'Archivo de resultados: {raw_files[-1].name}')
    
    with open(raw_files[-1], 'r', encoding='utf-8') as f:
        raw_results = json.load(f)
    
    with open(summary_files[-1], 'r', encoding='utf-8') as f:
        summary = json.load(f)
    
    df = pd.DataFrame(raw_results)
    df_summary = pd.DataFrame(summary).T
    df_summary.index.name = 'modelo'
    
    print(f'Total de evaluaciones: {len(df)}')
    print(f'Modelos evaluados: {df["model_name"].nunique()}')
    display(df_summary)

## 2. Comparacion de Latencia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras: latencia promedio
valid_df = df[df['latency_seconds'] >= 0]
latency_avg = valid_df.groupby('model_name')['latency_seconds'].mean().sort_values()
latency_avg.plot(kind='barh', ax=axes[0], color=sns.color_palette('husl', len(latency_avg)))
axes[0].set_xlabel('Latencia promedio (segundos)')
axes[0].set_title('Latencia Promedio por Modelo')

# Boxplot: distribucion de latencia
valid_df.boxplot(column='latency_seconds', by='model_name', ax=axes[1], vert=False)
axes[1].set_xlabel('Latencia (segundos)')
axes[1].set_title('Distribucion de Latencia por Modelo')
plt.suptitle('')

plt.tight_layout()
plt.savefig('results/latencia_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Precision de Retrieval

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

retrieval_acc = valid_df.groupby('model_name')['retrieval_hit'].mean().sort_values()
colors = ['#e74c3c' if v < 0.7 else '#f39c12' if v < 0.85 else '#2ecc71' for v in retrieval_acc]
retrieval_acc.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Precision de Retrieval')
ax.set_title('Precision de Recuperacion por Modelo')
ax.set_xlim(0, 1)

for i, v in enumerate(retrieval_acc):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center')

plt.tight_layout()
plt.savefig('results/retrieval_precision.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Relevancia y Fidelidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Relevancia promedio
relevancy = valid_df.groupby('model_name')['answer_relevancy'].mean().sort_values()
relevancy.plot(kind='barh', ax=axes[0], color=sns.color_palette('coolwarm', len(relevancy)))
axes[0].set_xlabel('Relevancia Promedio')
axes[0].set_title('Relevancia de Respuestas')
axes[0].set_xlim(0, 1)

# Fidelidad promedio
faithfulness = valid_df.groupby('model_name')['faithfulness'].mean().sort_values()
faithfulness.plot(kind='barh', ax=axes[1], color=sns.color_palette('coolwarm', len(faithfulness)))
axes[1].set_xlabel('Fidelidad Promedio')
axes[1].set_title('Fidelidad al Contexto')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('results/relevancia_fidelidad.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scatter: Latencia vs Calidad (Frontera de Pareto)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

models = valid_df['model_name'].unique()
colors = sns.color_palette('husl', len(models))

for i, model in enumerate(models):
    model_df = valid_df[valid_df['model_name'] == model]
    avg_latency = model_df['latency_seconds'].mean()
    avg_quality = (model_df['answer_relevancy'].mean() + model_df['faithfulness'].mean()) / 2
    
    ax.scatter(avg_latency, avg_quality, s=200, c=[colors[i]], 
               label=model, zorder=5, edgecolors='black', linewidth=1)
    ax.annotate(model, (avg_latency, avg_quality), 
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Latencia Promedio (segundos)')
ax.set_ylabel('Calidad Promedio (relevancia + fidelidad) / 2')
ax.set_title('Latencia vs Calidad - Comparacion de Modelos')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('results/pareto_latencia_calidad.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Radar Chart: Comparacion Multi-dimensional

In [ ]:
metrics_for_radar = ['retrieval_accuracy', 'avg_answer_relevancy', 
                      'avg_faithfulness', 'no_answer_accuracy']
labels = ['Retrieval', 'Relevancia', 'Fidelidad', 'No-Respuesta']

# Agregar metrica inversa de latencia (velocidad normalizada)
if not df_summary.empty:
    max_lat = df_summary['avg_latency_seconds'].max()
    df_summary['speed_score'] = 1 - (df_summary['avg_latency_seconds'] / max_lat)
    metrics_for_radar.append('speed_score')
    labels.append('Velocidad')

    num_vars = len(labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    colors = sns.color_palette('husl', len(df_summary))

    for i, (model, row) in enumerate(df_summary.iterrows()):
        values = [row.get(m, 0) for m in metrics_for_radar]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors[i])
        ax.fill(angles, values, alpha=0.1, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_title('Comparacion Multi-dimensional de Modelos', y=1.08)
    ax.legend(bbox_to_anchor=(1.3, 1.1))

    plt.tight_layout()
    plt.savefig('results/radar_comparacion.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Heatmap de Metricas

In [ ]:
if not df_summary.empty:
    heatmap_cols = ['avg_latency_seconds', 'retrieval_accuracy', 
                    'avg_answer_relevancy', 'avg_faithfulness', 
                    'hallucination_rate', 'no_answer_accuracy']
    heatmap_labels = ['Latencia (s)', 'Retrieval', 'Relevancia', 
                      'Fidelidad', 'Alucinacion', 'No-Respuesta']
    
    existing_cols = [c for c in heatmap_cols if c in df_summary.columns]
    existing_labels = [heatmap_labels[heatmap_cols.index(c)] for c in existing_cols]
    
    heatmap_data = df_summary[existing_cols].copy()
    heatmap_data.columns = existing_labels
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(heatmap_data) * 1.2)))
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
                ax=ax, linewidths=0.5, center=0.5)
    ax.set_title('Heatmap de Metricas por Modelo')
    
    plt.tight_layout()
    plt.savefig('results/heatmap_metricas.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Analisis por Dificultad

In [ ]:
if 'difficulty' in valid_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Fidelidad por dificultad y modelo
    pivot_faith = valid_df.pivot_table(
        values='faithfulness', index='model_name', 
        columns='difficulty', aggfunc='mean'
    )
    pivot_faith.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Fidelidad por Dificultad')
    axes[0].set_ylabel('Fidelidad')
    axes[0].legend(title='Dificultad')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Latencia por dificultad y modelo
    pivot_lat = valid_df.pivot_table(
        values='latency_seconds', index='model_name', 
        columns='difficulty', aggfunc='mean'
    )
    pivot_lat.plot(kind='bar', ax=axes[1])
    axes[1].set_title('Latencia por Dificultad')
    axes[1].set_ylabel('Latencia (s)')
    axes[1].legend(title='Dificultad')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('results/analisis_dificultad.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Conclusion y Recomendacion

In [ ]:
if not df_summary.empty:
    # Calcular score compuesto
    # Pesos: relevancia 0.3, fidelidad 0.3, retrieval 0.2, velocidad 0.1, no_answer 0.1
    max_lat = df_summary['avg_latency_seconds'].max()
    df_summary['composite_score'] = (
        0.30 * df_summary['avg_answer_relevancy'] +
        0.30 * df_summary['avg_faithfulness'] +
        0.20 * df_summary['retrieval_accuracy'] +
        0.10 * (1 - df_summary['avg_latency_seconds'] / max_lat) +
        0.10 * df_summary['no_answer_accuracy']
    )
    
    ranked = df_summary.sort_values('composite_score', ascending=False)
    
    print('RANKING DE MODELOS (score compuesto):')
    print('='*60)
    for i, (model, row) in enumerate(ranked.iterrows()):
        print(f'  {i+1}. {model}: {row["composite_score"]:.4f}')
        print(f'     Latencia: {row["avg_latency_seconds"]:.2f}s | '
              f'Relevancia: {row["avg_answer_relevancy"]:.3f} | '
              f'Fidelidad: {row["avg_faithfulness"]:.3f}')
    
    best = ranked.index[0]
    print(f'\nMODELO RECOMENDADO: {best}')
    print(f'Score compuesto: {ranked.iloc[0]["composite_score"]:.4f}')